# Fine-Tuning: Parameter Selection via Grid Search

Systematic grid search for **Modified Newton** and **Truncated Newton** method parameters.

**Two-phase approach:**
1. **Phase 1** â€” Tune method parameters (Armijo backtracking, Cholesky modification, forcing terms) with a fixed stopping criterion (`GradNormAbsolute(1e-8)`). The stopping criterion does not affect the convergence trajectory â€” it only decides when to stop.
2. **Phase 2** â€” Analyze stopping criteria using 3 tolerance bands (rough / good / very good) adapted per criterion type.

Uses **exact derivatives only** (finite difference variants are studied separately in Assignment Section 3).

---

**Table of Contents**
1. [Setup & Configuration](#setup)
2. [Helpers & Infrastructure](#helpers)
3. [Phase 1: Modified Newton Grid Search](#phase1-mn)
4. [Phase 1: Truncated Newton Grid Search](#phase1-tn)
5. [Phase 1: Results & Visualization](#phase1-results)
6. [Phase 2: Stopping Criteria Analysis](#phase2)
7. [Export & Summary](#export)

<a id="setup"></a>
## 1. Setup & Configuration

In [ ]:
import sys
import time
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.functions import f16, f28, x_bar_16, x_bar_28
from src.gradients import grad_f16, grad_f28
from src.hessians import hess_f16, hess_f28
from src.methods.modified_newton import modified_newton
from src.methods.truncated_newton import truncated_newton
from src.starting_points import generate_starting_points
from src.stopping_criteria import (
    StoppingCriterion,
    GradNormAbsolute, GradNormRelative,
    FChangeAbsolute, FChangeRelative,
    XChangeAbsolute, XChangeRelative,
)

In [ ]:
# CONFIGURATION
SEED = min(346165, 323334)

TIME_LIMIT = 60

QUICK_MODE = False   # Set False for full experiment (1-4 hours)

if QUICK_MODE:
    DIMENSIONS     = [2, 1000]
    NUM_RANDOM     = 1          # x_bar + 1 random = 2 starting points
    MAX_ITER       = 200
else:
    DIMENSIONS     = [2, 1000, 10_000, 100_000]
    NUM_RANDOM     = 5          # x_bar + 5 random = 6 starting points
    MAX_ITER       = 1000

OOM_THRESHOLD_MB = 4096         # skip cells where dense Hessian exceeds 4 GB

PROBLEMS = {
    'P16': dict(f=f16, grad=grad_f16, hess=hess_f16,
                x_bar=x_bar_16, desc='Banded Trigonometric (diagonal H)'),
    'P28': dict(f=f28, grad=grad_f28, hess=hess_f28,
                x_bar=x_bar_28, desc='Variably Dimensioned (dense H)'),
}

print(f"SEED           = {SEED}")
print(f"QUICK_MODE     = {QUICK_MODE}")
print(f"DIMENSIONS     = {DIMENSIONS}")
print(f"Starting pts   = {1 + NUM_RANDOM} per (problem, n)")
print(f"MAX_ITER       = {MAX_ITER}")
print(f"OOM threshold  = {OOM_THRESHOLD_MB} MB")


<a id="helpers"></a>
## 2. Helpers & Infrastructure

In [ ]:
# Combined Stopping Criterion (OR logic)

class CombinedStoppingCriterion(StoppingCriterion):
    """OR-combination: fires when ANY inner criterion fires."""

    def __init__(self, criteria: list):
        self.criteria = criteria
        self.tol = None
        self._triggered = "max_iter"

    @property
    def name(self):
        return self._triggered

    def initialize(self, x0, F0, g0):
        for c in self.criteria:
            c.initialize(x0, F0, g0)

    def should_stop(self, k, x, F, g, x_prev, F_prev) -> bool:
        for c in self.criteria:
            if c.should_stop(k, x, F, g, x_prev, F_prev):
                self._triggered = c.name
                return True
        return False

    def __repr__(self):
        return f"Combined({self.criteria})"

In [ ]:
# OOM guard

def expected_hessian_mb(n):
    """Memory for a dense (n, n) float64 matrix in MB."""
    return n * n * 8 / (1024 ** 2)

def should_skip(prob_id, n, method='any'):
    """Skip if the combination would OOM or be infeasible.

    P16 has a sparse diagonal Hessian -> never OOM.
    P28 has a dense Hessian -> skip Modified Newton at large n.
    Truncated Newton on P28 can use matrix-free Hv products.
    """
    if prob_id == 'P16':
        return False
    dense_mb = expected_hessian_mb(n)
    if method == 'truncated_newton':
        return False
    return dense_mb > OOM_THRESHOLD_MB

# Preview
print("OOM preview:")
for pid in ('P16', 'P28'):
    for n in DIMENSIONS:
        mb = expected_hessian_mb(n)
        mn_skip = should_skip(pid, n, 'modified_newton')
        tn_skip = should_skip(pid, n, 'truncated_newton')
        mn_tag = 'SKIP' if mn_skip else 'OK'
        tn_tag = 'SKIP' if tn_skip else 'OK'
        print(f"  {pid} n={n:>6d}: {mb:>10.0f} MB  MN={mn_tag:>4s}  TN={tn_tag:>4s}")



In [ ]:
# Experimental convergence rate estimator
# Estimates the convergence order p from the ||g_k|| sequence via
# successive log-ratios: p_k = log||g_{k+1}|| / log||g_k||.
# p~1 -> linear, p~1.5 -> superlinear, p~2 -> quadratic.
# This is the value reported as "rate" in the results tables.

def experimental_rate(g_norms):
    """Estimate convergence order p from ||g_k|| sequence.

    Model: ||g_{k+1}|| ~ C * ||g_k||^p
    p=1 linear, p~1.5 superlinear, p=2 quadratic.
    Returns median of tail estimates. NaN if not computable.
    """
    e = np.asarray(g_norms, dtype=float)
    if e.size < 4:
        return float('nan')
    e = e[e > 1e-14]
    if e.size < 4:
        return float('nan')
    log_e = np.log(e)
    num = np.diff(log_e)[1:]
    den = np.diff(log_e)[:-1]
    mask = (np.abs(den) > 1e-12) & np.isfinite(num) & np.isfinite(den)
    if not mask.any():
        return float('nan')
    p_vals = num[mask] / den[mask]
    p_tail = p_vals[-min(5, p_vals.size):]
    return float(np.median(p_tail))


# Aggregation helper

def aggregate_grid(df, param_cols):
    """Aggregate grid search results.

    1. Per (params, problem, n): mean/std across starting points.
    2. Average across (problem, n) with equal weight.
    3. Sort by (-success_rate, avg_iter).
    """
    group_inner = param_cols + ['problem', 'n'] 
    agg = df.groupby(group_inner).agg(
        success_rate=('success', 'mean'),
        mean_iter=('n_iter', 'mean'),
        std_iter=('n_iter', 'std'),
        mean_grad=('grad_norm', lambda s: np.nanmean(s)),
        mean_time=('time_s', 'mean'),
        n_runs=('success', 'count'),
    ).reset_index()

    summary = agg.groupby(param_cols).agg(
        avg_success=('success_rate', 'mean'),
        avg_iter=('mean_iter', 'mean'),
        avg_time=('mean_time', 'mean'),
        avg_grad=('mean_grad', lambda s: np.nanmean(s)),
        min_success=('success_rate', 'min'),
    ).reset_index()

    summary = summary.sort_values(
        by=['avg_success', 'avg_iter'], ascending=[False, True]
    ).reset_index(drop=True)
    summary.index.name = 'rank'
    return summary, agg

In [ ]:
# alpha_min analysis
# For each (rho, max_iter_backtrack) pair, the smallest achievable step is
# alpha_min = rho^T * alpha_0.  With alpha_0 = 1 this is just rho^T.
# Classification: 'safe' means alpha_min < eps_mach (~1e-15),
# i.e. backtracking can effectively reach zero; 'limited' otherwise.

rho_vals_preview = [0.3, 0.5, 0.8]
T_vals_preview = [30, 50, 100]

rows_alpha = []
for rho in rho_vals_preview:
    for T in T_vals_preview:
        a_min = rho ** T
        rows_alpha.append(dict(rho=rho, T=T, alpha_min=a_min,
                               log10_alpha=np.log10(a_min) if a_min > 0 else -np.inf))

alpha_df = pd.DataFrame(rows_alpha)

alpha_df['zone'] = alpha_df['alpha_min'].apply(
    lambda a: 'safe (< eps_mach)' if a < 1e-15 else 'limited')

print("Minimum achievable step: alpha_min = rho^T")
print(alpha_df.to_string(index=False))

Minimum achievable step: alpha_min = rho^T
 rho   T    alpha_min  log10_alpha              zone
 0.3  30 2.058911e-16   -15.686362 safe (< eps_mach)
 0.3  50 7.178980e-27   -26.143937 safe (< eps_mach)
 0.3 100 5.153775e-53   -52.287875 safe (< eps_mach)
 0.5  30 9.313226e-10    -9.030900           limited
 0.5  50 8.881784e-16   -15.051500 safe (< eps_mach)
 0.5 100 7.888609e-31   -30.103000 safe (< eps_mach)
 0.8  30 1.237940e-03    -2.907300           limited
 0.8  50 1.427248e-05    -4.845501           limited
 0.8 100 2.037036e-10    -9.691001           limited


In [ ]:
# Starting points generation

starts_cache = {}
rng = np.random.default_rng(SEED)

for prob_id in ('P16', 'P28'):
    x_bar_fn = PROBLEMS[prob_id]['x_bar']
    for n in DIMENSIONS:
        pts = generate_starting_points(x_bar_fn(n), num_random=NUM_RANDOM, rng=rng)
        starts_cache[(prob_id, n)] = pts
        print(f"  {prob_id} n={n:>6d}: {len(pts)} points "
              f"(||x_bar||={np.linalg.norm(pts[0]):.4f})")

print(f"\nTotal: {sum(len(v) for v in starts_cache.values())} starting points cached")

  P16 n=     2: 6 points (||x_bar||=1.4142)
  P16 n=  1000: 6 points (||x_bar||=31.6228)
  P16 n= 10000: 6 points (||x_bar||=100.0000)
  P16 n=100000: 6 points (||x_bar||=316.2278)
  P28 n=     2: 6 points (||x_bar||=0.5000)
  P28 n=  1000: 6 points (||x_bar||=18.2437)
  P28 n= 10000: 6 points (||x_bar||=57.7307)


  P28 n=100000: 6 points (||x_bar||=182.5728)

Total: 48 starting points cached


<a id="phase1-mn"></a>
## 3. Phase 1: Modified Newton — Grid Search

**Fixed stopping criterion:** `GradNormAbsolute(1e-8)` ("good solution" band).

**Parameters tuned** (see `docs/tuning_analysis.md` for theoretical justification):
| Parameter | Values | Description |
|-----------|--------|-------------|
| `beta` | 1e-6, 1e-3 | Cholesky modification heuristic |
| `rho` | 0.5, 0.8 | Backtracking reduction factor |

**Fixed parameters** (theoretically insensitive — see analysis):
| Parameter | Value | Reason |
|-----------|-------|--------|
| `alpha0` | 1 | Newton step, prerequisite for quadratic convergence |
| `c1` | 1e-4 | Nocedal-Wright standard; 5000:1 margin on full Newton step |
| `max_tau_iter` | 100 | Safety; ≤25 doublings ever needed |
| `max_iter_backtrack` | 50 | Safety; 50 steps → α ≈ 1e-15 |

In [ ]:
# Modified Newton parameter grid (reduced — see docs/tuning_analysis.md)

MN_GRID = dict(
    beta = [1e-6, 1e-3],
    rho  = [0.5, 0.8],
)

# Fixed Armijo & safety parameters
MN_FIXED = dict(c1=1e-4, max_tau_iter=100, max_iter_backtrack=50)

mn_keys = list(MN_GRID.keys())
mn_combos = list(itertools.product(*MN_GRID.values()))

n_valid_cells = sum(
    len(starts_cache[(pid, n)])
    for pid in ('P16', 'P28') for n in DIMENSIONS
    if not should_skip(pid, n, 'modified_newton')
)

print(f"Grid: {' x '.join(str(len(v)) for v in MN_GRID.values())} "
      f"= {len(mn_combos)} combos")
print(f"Valid (problem, n, start) cells: {n_valid_cells}")
print(f"Total runs: {len(mn_combos) * n_valid_cells}")

In [ ]:
# Modified Newton grid search execution

mn_rows = []
total = len(mn_combos) * n_valid_cells
idx = 0
t0_total = time.perf_counter()
print(f"Modified Newton: {total} experiments ({len(mn_combos)} combos x {n_valid_cells} cells)")
print("-" * 80)

for combo in mn_combos:
    beta, rho = combo

    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        for n in DIMENSIONS:
            if should_skip(prob_id, n, 'modified_newton'):
                continue
            for si, x0 in enumerate(starts_cache[(prob_id, n)]):
                idx += 1
                stop = GradNormAbsolute(tol=1e-8)
                t0 = time.perf_counter()
                try:
                    res = modified_newton(
                        pinfo['f'], x0, stop,
                        grad_f=pinfo['grad'], hess_f=pinfo['hess'],
                        alpha0=1.0, c1=MN_FIXED['c1'], rho=rho,
                        beta=beta, max_tau_iter=MN_FIXED['max_tau_iter'],
                        max_iter=MAX_ITER,
                        max_iter_backtrack=MN_FIXED['max_iter_backtrack'],
                        return_history=False, time_limit=TIME_LIMIT)
                    elapsed = time.perf_counter() - t0
                    tag = 'OK' if res['success'] else 'FAIL'
                    print(f"[{idx:>4d}/{total}] {tag}  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} beta={beta:.0e} | "
                          f"it={res['n_iter']:>3d} ||g||={res['grad_norm']:.2e} "
                          f"t={elapsed:.2f}s")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        beta=beta, rho=rho,
                        n_iter=res['n_iter'], success=res['success'],
                        grad_norm=res['grad_norm'], f_star=res['f_star'],
                        time_s=elapsed,
                        stop_reason=res['stop_reason'],
                        chol_adj=res.get('n_chol_adjustments_total', 0))
                except MemoryError:
                    print(f"[{idx:>4d}/{total}] OOM  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} beta={beta:.0e}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        beta=beta, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason='memory_error', chol_adj=0)
                except Exception as e:
                    print(f"[{idx:>4d}/{total}] ERR  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} beta={beta:.0e} | {e}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        beta=beta, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason=f'err:{type(e).__name__}', chol_adj=0)
                mn_rows.append(row)

mn_df = pd.DataFrame(mn_rows)
elapsed_total = time.perf_counter() - t0_total
print("-" * 80)
print(f"Modified Newton DONE: {len(mn_df)} runs in {elapsed_total:.1f}s, "
      f"{mn_df['success'].sum()} successes "
      f"({mn_df['success'].mean()*100:.1f}%)")

In [ ]:
# Modified Newton: aggregation and ranking

mn_param_cols = ['beta', 'rho']
mn_summary, mn_detail = aggregate_grid(mn_df, mn_param_cols)

print("=== Modified Newton Configurations ===")
display(mn_summary)

best_mn = mn_summary.iloc[0]
print(f"\nBest Modified Newton:")
print(f"  beta={best_mn['beta']:.0e}, rho={best_mn['rho']}")
print(f"  success={best_mn['avg_success']:.2%}, "
      f"avg_iter={best_mn['avg_iter']:.1f}")

# Per-dimension detail for the best config
mask = pd.Series(True, index=mn_detail.index)
for col in mn_param_cols:
    mask &= mn_detail[col] == best_mn[col]
print("\n=== Per-dimension detail (best config) ===")
display(mn_detail[mask].sort_values(['problem', 'n']))

<a id="phase1-tn"></a>
## 4. Phase 1: Truncated Newton — Grid Search

**Fixed stopping criterion:** `GradNormAbsolute(1e-8)`.

**Parameters tuned** (see `docs/tuning_analysis.md`):
| Parameter | Values | Description |
|-----------|--------|-------------|
| `forcing` | superlinear, quadratic | Forcing sequence for inner CG tolerance $\eta_k$ |
| `rho` | 0.5, 0.8 | Backtracking reduction factor |

**Fixed parameters:**
| Parameter | Value | Reason |
|-----------|-------|--------|
| `alpha0` | 1 | Newton step |
| `c1` | 1e-4 | Standard; insensitive |
| `cg_max_iter` | None (=n) | CG converges in 1-2 iter for P16/P28 |
| `max_iter_backtrack` | 50 | Safety |

**Forcing sequences** (Theorem 6.2 in [SW]):
- `superlinear`: $\eta_k = \min(0.5, \sqrt{\|\nabla f(x_k)\|})$ → superlinear rate
- `quadratic`: $\eta_k = \min(0.5, \|\nabla f(x_k)\|)$ → quadratic rate
- `linear` excluded: constant $\eta=0.5$ → linear convergence, never competitive

In [ ]:
# Truncated Newton parameter grid (reduced — see docs/tuning_analysis.md)

TN_GRID = dict(
    forcing = ['superlinear', 'quadratic'],
    rho     = [0.5, 0.8],
)

# Fixed Armijo & CG parameters
TN_FIXED = dict(c1=1e-4, cg_max_iter=None, max_iter_backtrack=50)

tn_keys = list(TN_GRID.keys())
tn_combos = list(itertools.product(*TN_GRID.values()))

tn_valid_cells = sum(
    len(starts_cache[(pid, n)])
    for pid in ('P16', 'P28') for n in DIMENSIONS
    if not should_skip(pid, n, 'truncated_newton')
)

print(f"Grid: {' x '.join(str(len(v)) for v in TN_GRID.values())} "
      f"= {len(tn_combos)} combos")
print(f"Valid (problem, n, start) cells: {tn_valid_cells}")
print(f"Total runs: {len(tn_combos) * tn_valid_cells}")

In [ ]:
# Truncated Newton grid search execution

tn_rows = []
total_tn = len(tn_combos) * tn_valid_cells
idx = 0
t0_total = time.perf_counter()
print(f"Truncated Newton: {total_tn} experiments ({len(tn_combos)} combos x {tn_valid_cells} cells)")
print("-" * 80)

for combo in tn_combos:
    forcing, rho = combo

    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        for n in DIMENSIONS:
            if should_skip(prob_id, n, 'truncated_newton'):
                continue
            for si, x0 in enumerate(starts_cache[(prob_id, n)]):
                idx += 1
                stop = GradNormAbsolute(tol=1e-8)
                t0 = time.perf_counter()
                try:
                    res = truncated_newton(
                        pinfo['f'], x0, stop,
                        grad_f=pinfo['grad'],
                        hess_f=(pinfo['hess'] if expected_hessian_mb(n) <= OOM_THRESHOLD_MB else None),
                        alpha0=1.0, c1=TN_FIXED['c1'], rho=rho,
                        forcing=forcing, cg_max_iter=TN_FIXED['cg_max_iter'],
                        max_iter=MAX_ITER,
                        max_iter_backtrack=TN_FIXED['max_iter_backtrack'],
                        return_history=False, time_limit=TIME_LIMIT)
                    elapsed = time.perf_counter() - t0
                    tag = 'OK' if res['success'] else 'FAIL'
                    print(f"[{idx:>4d}/{total_tn}] {tag}  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} f={forcing[:5]} | "
                          f"it={res['n_iter']:>3d} ||g||={res['grad_norm']:.2e} "
                          f"cg_tot={res.get('cg_iters_total',0)} "
                          f"t={elapsed:.2f}s")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        forcing=forcing, rho=rho,
                        n_iter=res['n_iter'], success=res['success'],
                        grad_norm=res['grad_norm'], f_star=res['f_star'],
                        time_s=elapsed,
                        stop_reason=res['stop_reason'],
                        cg_total=res.get('cg_iters_total', 0),
                        neg_curv=res.get('neg_curvature_count', 0))
                except MemoryError:
                    print(f"[{idx:>4d}/{total_tn}] OOM  {prob_id} n={n:<6d} s={si}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        forcing=forcing, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason='memory_error',
                        cg_total=0, neg_curv=0)
                except Exception as e:
                    print(f"[{idx:>4d}/{total_tn}] ERR  {prob_id} n={n:<6d} s={si} | {e}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        forcing=forcing, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason=f'err:{type(e).__name__}',
                        cg_total=0, neg_curv=0)
                tn_rows.append(row)

tn_df = pd.DataFrame(tn_rows)
elapsed_total = time.perf_counter() - t0_total
print("-" * 80)
print(f"Truncated Newton DONE: {len(tn_df)} runs in {elapsed_total:.1f}s, "
      f"{tn_df['success'].sum()} successes "
      f"({tn_df['success'].mean()*100:.1f}%)")

In [ ]:
# Truncated Newton: aggregation and ranking

tn_param_cols = ['forcing', 'rho']
tn_summary, tn_detail = aggregate_grid(tn_df, tn_param_cols)

print("=== Truncated Newton Configurations ===")
display(tn_summary)

best_tn = tn_summary.iloc[0]
print(f"\nBest Truncated Newton:")
print(f"  forcing={best_tn['forcing']}, rho={best_tn['rho']}")
print(f"  success={best_tn['avg_success']:.2%}, "
      f"avg_iter={best_tn['avg_iter']:.1f}")

# Per-dimension detail for the best config
mask = pd.Series(True, index=tn_detail.index)
for col in tn_param_cols:
    mask &= tn_detail[col] == best_tn[col]
print("\n=== Per-dimension detail (best config) ===")
display(tn_detail[mask].sort_values(['problem', 'n']))

In [ ]:
# Convergence rate for best configs (n=2 with history)

rate_rows = []
for method_name, method_fn, params in [
    ('ModNewton', modified_newton,
     dict(beta=best_mn['beta'], rho=best_mn['rho'], **MN_FIXED)),
    ('TruncNewton', truncated_newton,
     dict(forcing=best_tn['forcing'], rho=best_tn['rho'], **TN_FIXED)),
]:
    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        x0 = starts_cache[(prob_id, 2)][0]
        stop = GradNormAbsolute(tol=1e-8)
        res = method_fn(pinfo['f'], x0, stop,
                        grad_f=pinfo['grad'], hess_f=pinfo['hess'],
                        alpha0=1.0, max_iter=MAX_ITER,
                        return_history=True, **params)
        g_norms = [h['grad_norm'] for h in res.get('history', [])]
        rate = experimental_rate(g_norms)
        rate_rows.append(dict(method=method_name, problem=prob_id,
                              n_iter=res['n_iter'], rate=rate))

print("=== Convergence Rate Estimate (n=2, x_bar) ===")
print(pd.DataFrame(rate_rows).to_string(index=False))

<a id="phase1-results"></a>
## 5. Phase 1: Results & Visualization

In [ ]:
# Best parameters summary

summary_rows = [
    {
        'Method': 'Modified Newton',
        'rho': best_mn['rho'],
        'Method-specific': f"beta={best_mn['beta']:.0e}",
        'Fixed': f"c1=1e-4, max_tau=100, max_bt=50",
        'Success Rate': f"{best_mn['avg_success']:.2%}",
        'Avg Iterations': f"{best_mn['avg_iter']:.1f}",
    },
    {
        'Method': 'Truncated Newton',
        'rho': best_tn['rho'],
        'Method-specific': f"forcing={best_tn['forcing']}",
        'Fixed': f"c1=1e-4, cg_max=n, max_bt=50",
        'Success Rate': f"{best_tn['avg_success']:.2%}",
        'Avg Iterations': f"{best_tn['avg_iter']:.1f}",
    },
]

display(Markdown("### Selected Parameters"))
display(pd.DataFrame(summary_rows).set_index('Method'))

In [ ]:
# Heatmaps: success rate for the tuned parameters

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MN: beta vs rho
ax = axes[0]
pivot = mn_df.groupby(['beta', 'rho'])['success'].mean().unstack('rho')
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{v}' for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{v:.0e}' for v in pivot.index])
ax.set_xlabel('rho')
ax.set_ylabel('beta')
ax.set_title('Modified Newton — Success Rate')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i, j]:.0%}',
                ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# TN: forcing vs rho
ax = axes[1]
pivot = tn_df.groupby(['forcing', 'rho'])['success'].mean().unstack('rho')
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{v}' for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{v}' for v in pivot.index])
ax.set_xlabel('rho')
ax.set_ylabel('forcing')
ax.set_title('Truncated Newton — Success Rate')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i, j]:.0%}',
                ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Success Rate — Reduced Grid', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# All configurations: bar chart (avg iterations)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (summary, title, pcols) in zip(axes, [
    (mn_summary, 'Modified Newton', mn_param_cols),
    (tn_summary, 'Truncated Newton', tn_param_cols),
]):
    labels = [", ".join(f"{c}={row[c]}" for c in pcols)
             for _, row in summary.iterrows()]
    colors = ['#2ecc71' if s == 1.0 else '#e74c3c'
             for s in summary['avg_success']]
    ax.barh(labels, summary['avg_iter'], color=colors)
    ax.set_xlabel('Avg iterations')
    ax.set_title(f'{title}\n(green = 100% success, red = <100%)')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Per-(problem, n) detail for best configs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (detail, best, title, pcols) in zip(axes, [
    (mn_detail, best_mn, 'Modified Newton', mn_param_cols),
    (tn_detail, best_tn, 'Truncated Newton', tn_param_cols),
]):
    mask = pd.Series(True, index=detail.index)
    for col in pcols:
        mask &= detail[col] == best[col]
    sub = detail[mask].sort_values(['problem', 'n'])
    labels = [f"{r['problem']} n={r['n']}"
             for _, r in sub.iterrows()]
    colors = ['#2ecc71' if s == 1.0 else '#e74c3c'
             for s in sub['success_rate']]
    ax.barh(labels, sub['mean_iter'], color=colors)
    ax.set_xlabel('Mean iterations')
    ax.set_title(f'{title} (best config)')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

<a id="phase2"></a>
## 6. Phase 2: Stopping Criteria Analysis (Post-hoc)

The stopping criterion does **not** affect the convergence trajectory — it only decides when to declare convergence.

**Efficient approach**: run each algorithm **once** with `return_history=True` and a tight tolerance, then evaluate all stopping criteria **post-hoc** on the recorded trajectory. This eliminates the 22× re-run overhead.

### Tolerance Bands (from course slides)

| Band | TOL | Quality |
|------|-----|---------|
| **Rough** | $10^{-4}$ | Rough precision |
| **Good** | $10^{-8}$ | Good solution |
| **Very good** | $10^{-12}$ | Very demanding, often unnecessary |

### Criteria evaluated

| # | Criterion | Band | Purpose |
|---|-----------|------|---------|
| 1 | `grad_abs` | rough / good / very_good | Gold standard |
| 2 | `x_abs` | rough / good | Secondary confirmation |
| 3 | `grad_rel` | rough | Show it works only at loose tolerance |
| 4 | `combined_abs` | good | Combined (OR) approach |

### How to read the results

For each (method, problem, n, start), the algorithm history records `grad_norm`, `f`, `x`, `x_prev` at each iteration. We scan the history and find **the first iteration where each criterion would fire**. This gives:

- **`stop_iter`**: iteration at which the criterion would stop
- **`grad_norm_at_stop`**: quality of the solution at that point
- **`success`**: whether the criterion fired before `max_iter`

A **good** stopping criterion:
1. **Fires** (success = True) — it actually detects convergence
2. Fires at a point with **small `grad_norm_at_stop`** — the solution is accurate
3. Fires **early** (low `stop_iter`) — it does not waste iterations beyond convergence

A **bad** criterion (e.g. `grad_rel` at tight tolerance on P28) either:
- Fires too early with **huge** `grad_norm_at_stop` (false convergence)
- Never fires (the relative threshold is met trivially or never)

In [ ]:
# Tolerance bands adapted per criterion type

TOLERANCE_BANDS = {
    #              (tol_grad,  tol_f,    tol_x)
    'rough':       (1e-4,      1e-8,     1e-4),
    'good':        (1e-8,      1e-16,    1e-8),
    'very_good':   (1e-12,     None,     1e-12),  # None = f-change not feasible
}

def make_stopping_config(crit_type, band):
    """Create StoppingCriterion for a given type and band.

    Returns None if the combination is not feasible.
    """
    tol_g, tol_f, tol_x = TOLERANCE_BANDS[band]
    factories = {
        'grad_abs':     (tol_g, lambda t: GradNormAbsolute(t)),
        'grad_rel':     (tol_g, lambda t: GradNormRelative(t)),
        'x_abs':        (tol_x, lambda t: XChangeAbsolute(t)),
        'combined_abs': (None,  lambda _: CombinedStoppingCriterion([
                            GradNormAbsolute(tol_g),
                            *([FChangeAbsolute(tol_f)] if tol_f is not None else []),
                            XChangeAbsolute(tol_x)])),
    }
    if crit_type not in factories:
        return None
    tol, factory = factories[crit_type]
    return factory(tol)

# Reduced set: 7 (crit_type, band) pairs (from 22)
SC_CONFIGS = [
    ('grad_abs',     'rough'),
    ('grad_abs',     'good'),
    ('grad_abs',     'very_good'),
    ('x_abs',        'rough'),
    ('x_abs',        'good'),
    ('grad_rel',     'rough'),
    ('combined_abs', 'good'),
]

print(f"{len(SC_CONFIGS)} stopping criterion configurations:")
for ct, b in SC_CONFIGS:
    tol_g, tol_f, tol_x = TOLERANCE_BANDS[b]
    print(f"  {ct:15s} @ {b:10s}  "
          f"(tol_g={tol_g:.0e}, tol_x={tol_x:.0e})")

In [ ]:
# Phase 2: post-hoc stopping criteria evaluation
#
# Strategy: run each (method, problem, n, start) ONCE with a wrapper
# that logs lightweight metrics (grad_norm, f, ||dx||) at each iteration.
# Then evaluate all SC_CONFIGS on the recorded log — zero extra runs.
#
# This avoids storing full x vectors (which would be 800 MB/run at n=100k).

class MetricsLogger(StoppingCriterion):
    """Wraps a tight stopping criterion and logs per-iteration metrics."""

    def __init__(self, inner_stop):
        self.inner = inner_stop
        self.tol = inner_stop.tol
        self.log = []

    @property
    def name(self):
        return self.inner.name

    def initialize(self, x0, F0, g0):
        self.inner.initialize(x0, F0, g0)
        g0_norm = float(np.linalg.norm(g0))
        self.log = [{'k': 0, 'grad_norm': g0_norm, 'f': float(F0),
                     'x_change': 0.0}]

    def should_stop(self, k, x, F, g, x_prev, F_prev):
        self.log.append({
            'k': k,
            'grad_norm': float(np.linalg.norm(g)),
            'f': float(F),
            'x_change': float(np.linalg.norm(x - x_prev)) if k > 0 else 0.0,
        })
        return self.inner.should_stop(k, x, F, g, x_prev, F_prev)


def eval_criterion_on_log(log, crit_type, band):
    """Evaluate a stopping criterion on a lightweight metrics log.

    Returns (stop_iter, grad_norm_at_stop, f_at_stop, reason) or None.
    """
    tol_g, tol_f, tol_x = TOLERANCE_BANDS[band]
    g0_norm = log[0]['grad_norm'] if log else 1.0

    for entry in log[1:]:  # skip k=0 (no previous iterate)
        k = entry['k']
        gn = entry['grad_norm']
        f_val = entry['f']
        dx = entry['x_change']
        f_prev = log[k - 1]['f'] if k > 0 else f_val

        fired = False
        reason = ''

        if crit_type == 'grad_abs':
            fired = gn <= tol_g
            reason = 'grad_abs'
        elif crit_type == 'grad_rel':
            fired = (gn / max(g0_norm, 1e-300)) <= tol_g
            reason = 'grad_rel'
        elif crit_type == 'x_abs':
            fired = dx <= tol_x
            reason = 'x_abs'
        elif crit_type == 'combined_abs':
            if gn <= tol_g:
                fired, reason = True, 'grad_abs'
            elif tol_f is not None and abs(f_val - f_prev) <= tol_f:
                fired, reason = True, 'f_abs'
            elif dx <= tol_x:
                fired, reason = True, 'x_abs'

        if fired:
            return k, gn, f_val, reason

    return None


# Step 1: run each cell once with MetricsLogger wrapping GradNormAbsolute(1e-12)

best_mn_params = dict(beta=best_mn['beta'], rho=best_mn['rho'], **MN_FIXED)
best_tn_params = dict(forcing=best_tn['forcing'], rho=best_tn['rho'], **TN_FIXED)

methods_config = [
    ('ModNewton',  modified_newton,  best_mn_params),
    ('TruncNewton', truncated_newton, best_tn_params),
]

metrics_logs = {}
idx = 0
total_runs = sum(
    len(starts_cache[(pid, n)])
    for mn, _, _ in methods_config
    for pid in ('P16', 'P28')
    for n in DIMENSIONS
    if not should_skip(pid, n,
        'modified_newton' if 'Mod' in mn else 'truncated_newton')
)
t0_total = time.perf_counter()
print(f"Phase 2: collecting metrics — {total_runs} runs")
print("-" * 80)

for method_name, method_fn, params in methods_config:
    m_key = 'modified_newton' if 'Mod' in method_name else 'truncated_newton'
    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        for n in DIMENSIONS:
            if should_skip(prob_id, n, m_key):
                continue
            for si, x0 in enumerate(starts_cache[(prob_id, n)]):
                idx += 1
                logger = MetricsLogger(GradNormAbsolute(tol=1e-12))
                t0 = time.perf_counter()
                try:
                    res = method_fn(
                        pinfo['f'], x0, logger,
                        grad_f=pinfo['grad'],
                        hess_f=(pinfo['hess']
                                if m_key == 'modified_newton'
                                or expected_hessian_mb(n) <= OOM_THRESHOLD_MB
                                else None),
                        alpha0=1.0, max_iter=MAX_ITER,
                        return_history=False, time_limit=TIME_LIMIT,
                        **params)
                    elapsed = time.perf_counter() - t0
                    tag = 'OK' if res['success'] else 'FAIL'
                    print(f"[{idx:>4d}/{total_runs}] {tag}  {method_name:11s} "
                          f"{prob_id} n={n:<6d} s={si} | "
                          f"it={res['n_iter']:>3d} ||g||={res['grad_norm']:.2e} "
                          f"t={elapsed:.2f}s")
                    metrics_logs[(method_name, prob_id, n, si)] = logger.log
                except Exception as e:
                    print(f"[{idx:>4d}/{total_runs}] ERR  {method_name:11s} "
                          f"{prob_id} n={n:<6d} s={si} | {e}")
                    metrics_logs[(method_name, prob_id, n, si)] = []

print("-" * 80)
print(f"Metrics collected in {time.perf_counter() - t0_total:.1f}s")

# Step 2: evaluate all criteria post-hoc on every metrics log
sc_rows = []
for (method_name, prob_id, n, si), log in metrics_logs.items():
    for crit_type, band in SC_CONFIGS:
        result = eval_criterion_on_log(log, crit_type, band)
        if result is not None:
            stop_iter, grad_at_stop, f_at_stop, reason = result
            sc_rows.append(dict(
                method=method_name, problem=prob_id, n=n,
                start_idx=si, crit_type=crit_type, band=band,
                stop_iter=stop_iter, success=True,
                grad_norm_at_stop=grad_at_stop,
                f_at_stop=f_at_stop,
                stop_reason=reason))
        else:
            final_g = log[-1]['grad_norm'] if log else np.nan
            final_f = log[-1]['f'] if log else np.nan
            sc_rows.append(dict(
                method=method_name, problem=prob_id, n=n,
                start_idx=si, crit_type=crit_type, band=band,
                stop_iter=len(log) - 1, success=False,
                grad_norm_at_stop=final_g,
                f_at_stop=final_f,
                stop_reason='max_iter'))

sc_df = pd.DataFrame(sc_rows)
print(f"\nPost-hoc evaluation: {len(sc_df)} entries "
      f"({len(metrics_logs)} runs x {len(SC_CONFIGS)} criteria)")

In [ ]:
# Stopping criteria: aggregation and tables

sc_agg = sc_df.groupby(['method', 'crit_type', 'band']).agg(
    success_rate=('success', 'mean'),
    mean_stop_iter=('stop_iter', 'mean'),
    std_stop_iter=('stop_iter', 'std'),
    mean_grad_at_stop=('grad_norm_at_stop', lambda s: np.nanmean(s)),
    mean_f_at_stop=('f_at_stop', lambda s: np.nanmean(s)),
).reset_index()

for method in ('ModNewton', 'TruncNewton'):
    print(f"\n=== Stopping Criteria — {method} ===")
    sub = sc_agg[sc_agg['method'] == method].sort_values(['band', 'crit_type'])
    print(sub.to_string(index=False))

print("\n--- Interpretation guide ---")
print("A good stopping criterion has:")
print("  1. success_rate = 1.0  (it actually fires)")
print("  2. small mean_grad_at_stop  (accurate solution when it fires)")
print("  3. low mean_stop_iter  (does not waste iterations past convergence)")
print("\nWatch for: grad_rel at tight tolerance on P28 →")
print("  fires early with huge grad_norm (false convergence due to ||g0|| = O(n^7))")

In [ ]:
# Stopping criteria: visualization

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, method in zip(axes, ['ModNewton', 'TruncNewton']):
    sub = sc_agg[sc_agg['method'] == method].copy()
    sub['label'] = sub['crit_type'] + " @ " + sub['band']
    colors = ['#2ecc71' if g < 1e-3 else '#e74c3c'
              for g in sub['mean_grad_at_stop']]
    bars = ax.barh(sub['label'], sub['mean_stop_iter'], color=colors)
    ax.set_xlabel('Mean iterations to stop')
    ax.set_title(f'{method}\n(green = ||g|| < 1e-3, red = ||g|| ≥ 1e-3)')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

<a id="export"></a>
## 7. Export & Summary

In [ ]:
# Save results to CSV

results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)

mn_df.to_csv(results_dir / 'fine_tuning_modified_newton.csv', index=False)
tn_df.to_csv(results_dir / 'fine_tuning_truncated_newton.csv', index=False)
sc_df.to_csv(results_dir / 'fine_tuning_stopping_criteria.csv', index=False)

print(f"Saved to {results_dir}:")
print(f"  fine_tuning_modified_newton.csv   ({len(mn_df)} rows)")
print(f"  fine_tuning_truncated_newton.csv  ({len(tn_df)} rows)")
print(f"  fine_tuning_stopping_criteria.csv ({len(sc_df)} rows)")

In [ ]:
# Final summary

print("=" * 70)
print("FINE-TUNING SUMMARY")
print("=" * 70)
print(f"Dimensions: {DIMENSIONS}")
print(f"Starting points: {1 + NUM_RANDOM} per (problem, n)")
print(f"MAX_ITER: {MAX_ITER}")
print()
print("BEST MODIFIED NEWTON:")
print(f"  beta={best_mn['beta']:.0e}, rho={best_mn['rho']}")
print(f"  fixed: c1=1e-4, max_tau_iter=100, max_iter_backtrack=50")
print(f"  success rate: {best_mn['avg_success']:.2%}")
print(f"  avg iterations: {best_mn['avg_iter']:.1f}")
print()
print("BEST TRUNCATED NEWTON:")
print(f"  forcing={best_tn['forcing']}, rho={best_tn['rho']}")
print(f"  fixed: c1=1e-4, cg_max_iter=n, max_iter_backtrack=50")
print(f"  success rate: {best_tn['avg_success']:.2%}")
print(f"  avg iterations: {best_tn['avg_iter']:.1f}")
print()
print("=" * 70)